# Projet 1 — Analyse exploratoire des sous-types moléculaires du cancer du sein (TCGA-BRCA)

**Auteur :** [Ton nom]
**Date :** 2026
**Contact :** [email / LinkedIn / site portfolio]

---

## Résumé exécutif

> **À remplir en dernier — 4 à 6 lignes maximum.**
> Ce résumé doit pouvoir être lu en 30 secondes par un prospect non-data.
>
> Structure suggérée :
> - **Contexte** : pourquoi cette question importe cliniquement
> - **Approche** : quelles méthodes, sur quelles données
> - **Résultat clé** : 1 ou 2 chiffres marquants
> - **Implication** : pourquoi c'est utile pour la décision médicale

---

## Table des matières

1. [Contexte biologique & question de recherche](#1-contexte)
2. [Chargement et nettoyage des données](#2-nettoyage)
3. [Analyse exploratoire descriptive](#3-eda)
4. [Tests statistiques](#4-stats)
5. [Visualisations de synthèse](#5-viz)
6. [Discussion, limites et perspectives](#6-discussion)


<a id="1-contexte"></a>
## 1. Contexte biologique & question de recherche

### 1.1 Pourquoi le cancer du sein ?

Le cancer du sein est la première cause de cancer chez la femme dans le monde. C'est aussi une maladie **moléculairement hétérogène** : sous une apparence histologique parfois similaire, les tumeurs se répartissent en plusieurs **sous-types moléculaires intrinsèques** (classification PAM50) :

- **Luminal A** — récepteurs hormonaux positifs (ER+/PR+), HER2-, prolifération faible. Bon pronostic.
- **Luminal B** — récepteurs hormonaux positifs, prolifération plus élevée. Pronostic intermédiaire.
- **HER2-enriched** — surexpression de HER2. Agressif mais ciblable par thérapies anti-HER2.
- **Basal-like** (~triple-négatif) — ER-, PR-, HER2-. Agressif, options thérapeutiques limitées.
- **Normal-like** — profil proche du tissu mammaire sain.

Cette classification a des **implications cliniques directes** : choix du traitement, intensité du suivi, pronostic à 5 et 10 ans.

### 1.2 Question de recherche

> **Quelles caractéristiques cliniques distinguent les sous-types moléculaires du cancer du sein dans la cohorte TCGA-BRCA, et ces différences sont-elles statistiquement significatives après correction pour tests multiples ?**

### 1.3 Sous-questions opérationnelles

- Les sous-types diffèrent-ils par leur **âge au diagnostic** ?
- Diffèrent-ils par leur **stade tumoral** au diagnostic ?
- Diffèrent-ils par leur **statut vital** au dernier suivi ?
- Les **données manquantes** sont-elles distribuées au hasard, ou indiquent-elles un biais de recueil ?

### 1.4 Choix méthodologiques annoncés

- **Source de données** : TCGA-BRCA via le GDC Data Portal (portal.gdc.cancer.gov)
- **Type d'analyse** : exploratoire, descriptive et inférentielle (pas de modèle prédictif à ce stade)
- **Tests statistiques** : choisis en fonction de la nature des variables et de la normalité (justifiés dans la section 4)
- **Correction pour tests multiples** : Benjamini-Hochberg (FDR), seuil α=0.05


<a id="2-nettoyage"></a>
## 2. Chargement et nettoyage des données

### 2.1 Récupération des données

**Méthode recommandée pour ce projet** : télécharger les fichiers cliniques directement via le GDC Data Portal.

**Procédure étape par étape :**

1. Aller sur https://portal.gdc.cancer.gov/
2. Cliquer sur "Projects" → filtrer "Program: TCGA" → sélectionner "TCGA-BRCA"
3. Cliquer sur "Save New Cohort" → nommer (ex: `brca_project1`)
4. Aller dans "Cohort Builder" → vérifier les filtres (ici aucun, on prend toute la cohorte)
5. Cliquer sur "Repository" → onglet "Files" → filtrer :
   - Data Category: **Clinical**
   - Data Format: **bcr xml** ou **bcr biotab** (le format TSV est plus simple pour démarrer)
6. Cliquer sur "Add All Files to Cart" → "Cart" → "Download → Clinical (TSV)"

**Alternative plus rapide pour démarrer** : utiliser le package Python `cbio-py` ou `pycbio` qui interroge l'API cBioPortal.

**Pour les données moléculaires** (sous-type PAM50, expression génique) : à ajouter en complément depuis cBioPortal (étude `brca_tcga_pan_can_atlas_2018`).

### 2.2 Imports et configuration


In [ ]:
# Imports — bibliothèques standard de l'analyse de données scientifiques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Statistiques
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Configuration globale
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Style graphique cohérent (à réutiliser sur tout le portfolio)
sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.figsize'] = (8, 5)

# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chemins
DATA_DIR = Path('./data/raw')
PROCESSED_DIR = Path('./data/processed')
FIGURES_DIR = Path('./figures')

for d in [DATA_DIR, PROCESSED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Environnement prêt.")


### 2.3 Chargement du fichier clinique brut

Le GDC fournit plusieurs fichiers cliniques pour BRCA. Les principaux :
- `clinical.tsv` — données démographiques et diagnostiques (1 ligne par patient)
- `exposure.tsv` — expositions (tabac, alcool)
- `family_history.tsv`
- `follow_up.tsv` — **plusieurs lignes par patient** (un suivi peut-être répété dans le temps)

**Premier piège** : dans `follow_up.tsv`, un même patient peut apparaître plusieurs fois. Il faut garder la ligne la plus récente.


In [ ]:
# Charger le fichier clinique principal
# Le séparateur est une tabulation, et les valeurs manquantes sont codées de plusieurs façons
clinical_raw = pd.read_csv(
    DATA_DIR / 'clinical.tsv',
    sep='\t',
    na_values=["'--", '[Not Available]', '[Not Applicable]', '[Unknown]', '[Not Evaluated]', 'unknown']
)

print(f"Shape : {clinical_raw.shape}")
print(f"Nombre de patients uniques : {clinical_raw['case_submitter_id'].nunique()}")
clinical_raw.head()


### 2.4 Inventaire des variables et taux de remplissage

Avant tout nettoyage, on doit savoir ce qu'on a. Un bon réflexe : produire un tableau avec, pour chaque colonne, le **type**, le **nombre de valeurs uniques** et le **pourcentage de valeurs manquantes**.


In [ ]:
def column_inventory(df: pd.DataFrame) -> pd.DataFrame:
    """Retourne un inventaire des colonnes d'un DataFrame.

    Pour chaque colonne : type, nb de valeurs uniques, % de NaN.
    Utile pour rapidement identifier les colonnes à conserver / nettoyer / supprimer.
    """
    inventory = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_unique': df.nunique(dropna=True),
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().sum() / len(df) * 100).round(1),
    })
    return inventory.sort_values('pct_missing', ascending=False)

inventory = column_inventory(clinical_raw)
print(f"Total colonnes : {len(inventory)}")
print(f"Colonnes >50% manquantes : {(inventory['pct_missing'] > 50).sum()}")
inventory.head(20)


### 2.5 Sélection des variables d'intérêt

> **Décision de modélisation à documenter** : on conserve uniquement les variables pertinentes pour la question de recherche. Cela évite la tentation du "data dredging" (lancer des tests partout en espérant trouver quelque chose).

Variables retenues pour cette analyse :

| Variable | Type | Justification |
|---|---|---|
| `case_submitter_id` | identifiant | clé de jointure avec les données moléculaires |
| `age_at_diagnosis` | numérique (jours) | comparaison entre sous-types |
| `gender` | catégorielle | contrôle (cancer du sein masculin ≈1%) |
| `race`, `ethnicity` | catégorielles | détection de biais de cohorte |
| `vital_status` | catégorielle | "Alive" / "Dead" au dernier suivi |
| `ajcc_pathologic_stage` | ordinale | stade tumoral au diagnostic |
| `primary_diagnosis` | catégorielle | sous-type histologique |


In [ ]:
variables_of_interest = [
    'case_submitter_id',
    'age_at_diagnosis',
    'gender',
    'race',
    'ethnicity',
    'vital_status',
    'ajcc_pathologic_stage',
    'primary_diagnosis',
]

# On vérifie que toutes les colonnes existent (sécurité)
missing_cols = [c for c in variables_of_interest if c not in clinical_raw.columns]
if missing_cols:
    print(f"⚠️  Colonnes absentes : {missing_cols}")
else:
    print("✓ Toutes les colonnes d'intérêt sont présentes")

clinical = clinical_raw[variables_of_interest].copy()
clinical.head()


### 2.6 Gestion des doublons

Dans le fichier `clinical.tsv` du GDC, un même patient peut apparaître plusieurs fois si plusieurs prélèvements ou suivis ont été enregistrés.

**Règle de déduplication** : garder une seule ligne par `case_submitter_id`, en privilégiant la ligne la plus complète (le moins de NaN).


In [ ]:
print(f"Avant déduplication : {len(clinical)} lignes, {clinical['case_submitter_id'].nunique()} patients")

# Calcul du nombre de valeurs non-NaN par ligne, comme score de "complétude"
clinical['_completeness'] = clinical.notna().sum(axis=1)

# Trier par complétude décroissante puis garder la première ligne par patient
clinical = (
    clinical.sort_values('_completeness', ascending=False)
            .drop_duplicates(subset='case_submitter_id', keep='first')
            .drop(columns='_completeness')
            .reset_index(drop=True)
)

print(f"Après déduplication : {len(clinical)} lignes, {clinical['case_submitter_id'].nunique()} patients")


### 2.7 Conversion de l'âge

`age_at_diagnosis` est codé en **jours** dans TCGA. On convertit en années pour la lisibilité.


In [ ]:
clinical['age_years'] = clinical['age_at_diagnosis'] / 365.25

# Sanity check : distribution réaliste ?
print(clinical['age_years'].describe().round(1))


### 2.8 Nettoyage du stade tumoral

Le champ `ajcc_pathologic_stage` contient des sous-stades (IA, IB, IIA, etc.). Pour l'analyse, on les regroupe en stades principaux (I, II, III, IV).


In [ ]:
def simplify_stage(stage):
    """Regroupe les sous-stades AJCC en stades principaux."""
    if pd.isna(stage):
        return np.nan
    stage = str(stage).replace('Stage ', '').strip()
    # Stade X = non déterminable
    if stage in ('X', 'x', ''):
        return np.nan
    # Garder seulement les chiffres romains principaux
    for s in ['IV', 'III', 'II', 'I']:
        if stage.startswith(s):
            return s
    return np.nan

clinical['stage_simplified'] = clinical['ajcc_pathologic_stage'].apply(simplify_stage)

# Ordre logique des stades pour les visualisations
stage_order = ['I', 'II', 'III', 'IV']
clinical['stage_simplified'] = pd.Categorical(
    clinical['stage_simplified'], categories=stage_order, ordered=True
)

clinical['stage_simplified'].value_counts(dropna=False)


### 2.9 Fusion avec les sous-types moléculaires (PAM50)

> **À adapter** : cette étape suppose que tu as téléchargé séparément un fichier contenant la classification PAM50 (depuis cBioPortal, étude `brca_tcga_pan_can_atlas_2018`, colonne `Subtype` ou `SUBTYPE`).

Si tu n'as pas encore ce fichier, tu peux soit le télécharger maintenant, soit continuer l'analyse sans sous-types pour le moment et l'ajouter plus tard.


In [ ]:
# Charger les sous-types moléculaires
# Adapter le chemin et le nom de colonne selon ton fichier source
try:
    subtypes = pd.read_csv(DATA_DIR / 'pam50_subtypes.tsv', sep='\t')
    # Harmoniser l'identifiant : les barcodes cBioPortal ont parfois un suffixe -01
    subtypes['case_submitter_id'] = subtypes['SAMPLE_ID'].str[:12]
    subtypes = subtypes[['case_submitter_id', 'SUBTYPE']].drop_duplicates('case_submitter_id')

    # Jointure
    clinical = clinical.merge(subtypes, on='case_submitter_id', how='left')

    # Renommage pour lisibilité
    clinical = clinical.rename(columns={'SUBTYPE': 'pam50_subtype'})
    print(f"✓ Sous-types fusionnés. Couverture : {clinical['pam50_subtype'].notna().sum()}/{len(clinical)}")
    print(clinical['pam50_subtype'].value_counts(dropna=False))
except FileNotFoundError:
    print("⚠️  Fichier PAM50 non trouvé — l'analyse continuera sans cette variable.")
    print("    Télécharge `data_clinical_sample.txt` depuis cBioPortal (brca_tcga_pan_can_atlas_2018).")


### 2.10 Sauvegarde du jeu de données propre


In [ ]:
# Sauvegarde pour ne pas re-exécuter tout le nettoyage à chaque fois
clinical.to_csv(PROCESSED_DIR / 'brca_clinical_clean.csv', index=False)
print(f"✓ Données nettoyées sauvegardées ({len(clinical)} patients, {clinical.shape[1]} variables)")


<a id="3-eda"></a>
## 3. Analyse exploratoire descriptive

> **Principe directeur** : avant tout test statistique, on regarde les données. Chaque visualisation doit répondre à une question précise et être commentée.

### 3.1 Vue d'ensemble de la cohorte


In [ ]:
print(f"=== COHORTE TCGA-BRCA APRÈS NETTOYAGE ===")
print(f"Nombre de patients : {len(clinical)}")
print(f"Femmes / Hommes : {(clinical['gender'] == 'female').sum()} / {(clinical['gender'] == 'male').sum()}")
print(f"Âge médian au diagnostic : {clinical['age_years'].median():.1f} ans")
print(f"Étendue : {clinical['age_years'].min():.0f} - {clinical['age_years'].max():.0f} ans")
print(f"Décès observés : {(clinical['vital_status'] == 'Dead').sum()} ({(clinical['vital_status'] == 'Dead').mean()*100:.1f}%)")


### 3.2 Distribution de l'âge au diagnostic

**Question** : la distribution de l'âge est-elle approximativement normale ? (Cela conditionnera le choix des tests dans la section 4.)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme + densité
sns.histplot(clinical['age_years'].dropna(), bins=30, kde=True, ax=axes[0])
axes[0].set_title("Distribution de l'âge au diagnostic")
axes[0].set_xlabel("Âge (années)")
axes[0].set_ylabel("Nombre de patients")
axes[0].axvline(clinical['age_years'].median(), color='red', linestyle='--', label=f"Médiane : {clinical['age_years'].median():.1f}")
axes[0].legend()

# Q-Q plot pour évaluer la normalité
stats.probplot(clinical['age_years'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title("Q-Q plot vs loi normale")

plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_age_distribution.png', bbox_inches='tight')
plt.show()

# Test formel de normalité (Shapiro-Wilk)
stat, p = stats.shapiro(clinical['age_years'].dropna().sample(min(5000, clinical['age_years'].notna().sum()), random_state=42))
print(f"\nShapiro-Wilk : statistique={stat:.4f}, p={p:.2e}")
print("→ p < 0.05 : on rejette l'hypothèse de normalité." if p < 0.05
      else "→ p ≥ 0.05 : compatible avec une distribution normale.")


**Interprétation** :
> *À compléter après exécution.* En général sur TCGA-BRCA, la distribution d'âge est légèrement asymétrique avec une queue vers les âges jeunes. On utilisera donc des **tests non paramétriques** par défaut (Mann-Whitney, Kruskal-Wallis) plutôt que des t-tests.

### 3.3 Distribution par sous-type moléculaire


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))

    subtype_order = ['BRCA_LumA', 'BRCA_LumB', 'BRCA_Her2', 'BRCA_Basal', 'BRCA_Normal']
    available_subtypes = [s for s in subtype_order if s in clinical['pam50_subtype'].unique()]

    counts = clinical['pam50_subtype'].value_counts().reindex(available_subtypes)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='colorblind')

    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(int(v)), ha='center', fontweight='bold')

    ax.set_title("Effectifs par sous-type moléculaire PAM50")
    ax.set_xlabel("Sous-type")
    ax.set_ylabel("Nombre de patients")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_subtype_counts.png', bbox_inches='tight')
    plt.show()
else:
    print("Sous-types non disponibles — section ignorée.")


### 3.4 Âge selon le sous-type — première impression visuelle


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years',
        order=available_subtypes,
        palette='colorblind', ax=ax
    )
    sns.stripplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years',
        order=available_subtypes,
        color='black', alpha=0.2, size=2, ax=ax
    )
    ax.set_title("Âge au diagnostic selon le sous-type PAM50")
    ax.set_xlabel("Sous-type")
    ax.set_ylabel("Âge (années)")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_age_by_subtype.png', bbox_inches='tight')
    plt.show()


### 3.5 Stade tumoral selon le sous-type


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    crosstab = pd.crosstab(
        clinical['pam50_subtype'],
        clinical['stage_simplified'],
        normalize='index'
    ) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    crosstab.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')
    ax.set_title("Distribution du stade tumoral par sous-type (% au sein du sous-type)")
    ax.set_xlabel("Sous-type")
    ax.set_ylabel("% de patients")
    ax.legend(title='Stade', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_stage_by_subtype.png', bbox_inches='tight')
    plt.show()


<a id="4-stats"></a>
## 4. Tests statistiques

> **Principe directeur** : pour chaque test, on documente (1) la question, (2) le choix du test et sa justification, (3) les hypothèses sous-jacentes vérifiées, (4) l'interprétation.

### 4.1 L'âge diffère-t-il significativement entre les sous-types ?

**Question** : `age_years` (continue) vs `pam50_subtype` (catégorielle, >2 groupes).

**Choix du test** :
- Variable continue non normale (cf. section 3.2) → **test non paramétrique**
- Plus de 2 groupes → **Kruskal-Wallis** (équivalent non paramétrique de l'ANOVA)
- Si significatif → post-hoc **Dunn** avec correction Bonferroni


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    # Constituer les groupes
    groups = [
        clinical.loc[clinical['pam50_subtype'] == s, 'age_years'].dropna().values
        for s in available_subtypes
    ]

    # Test global
    stat_kw, p_kw = stats.kruskal(*groups)
    print(f"Kruskal-Wallis : H={stat_kw:.3f}, p={p_kw:.2e}")
    print(f"→ {'Différence significative détectée' if p_kw < 0.05 else 'Pas de différence significative'}")


### 4.2 Le stade tumoral est-il associé au sous-type ?

**Question** : `stage_simplified` (catégorielle ordinale) vs `pam50_subtype` (catégorielle).

**Choix du test** :
- Deux variables catégorielles → **test du Chi²** d'indépendance
- Vérifier que tous les effectifs théoriques ≥ 5 (sinon → test exact de Fisher)


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    contingency = pd.crosstab(clinical['pam50_subtype'], clinical['stage_simplified'])
    print("Tableau de contingence (effectifs observés) :")
    print(contingency)

    chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

    # Vérification de la condition d'application
    min_expected = expected.min()
    print(f"\nχ² = {chi2:.3f}, ddl = {dof}, p = {p_chi2:.2e}")
    print(f"Effectif théorique minimum : {min_expected:.1f}")

    if min_expected < 5:
        print("⚠️  Au moins un effectif théorique < 5 : utiliser le test exact de Fisher (via R ou statsmodels)")


### 4.3 La mortalité diffère-t-elle entre sous-types ?

**Question** : `vital_status` (binaire) vs `pam50_subtype` (catégorielle).

**Choix du test** : Chi² d'indépendance.

> **Note importante pour la suite** : cette analyse ne tient pas compte du temps de suivi. Une analyse de survie correcte (Kaplan-Meier, log-rank, Cox) sera l'objet du **Projet 4**. Ici on fait un proxy simpliste, et on le dit explicitement.


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    contingency_vital = pd.crosstab(clinical['pam50_subtype'], clinical['vital_status'])
    print(contingency_vital)

    chi2, p, dof, _ = stats.chi2_contingency(contingency_vital)
    print(f"\nχ² = {chi2:.3f}, ddl = {dof}, p = {p:.2e}")


### 4.4 Correction pour tests multiples

On a réalisé 3 tests sur la même cohorte. La probabilité d'obtenir au moins un faux positif à α=0.05 est : 1 − 0.95³ ≈ 14%. On applique donc une correction.

**Méthode choisie** : Benjamini-Hochberg (FDR). Plus puissante que Bonferroni, adaptée quand on s'attend à plusieurs vrais positifs.


In [ ]:
# À adapter avec les vraies p-values obtenues plus haut
results = pd.DataFrame({
    'test': ['Âge ~ sous-type', 'Stade ~ sous-type', 'Mortalité ~ sous-type'],
    'p_value': [p_kw, p_chi2, p],  # noms des variables tels que définis ci-dessus
})

rejected, p_corrected, _, _ = multipletests(results['p_value'], alpha=0.05, method='fdr_bh')
results['p_corrected_fdr'] = p_corrected
results['significant_fdr'] = rejected

print(results.to_string(index=False))


**Interprétation** :
> *À compléter après exécution.* Indiquer pour chaque test si la conclusion change après correction FDR, et formuler la conclusion en langage clinique (pas seulement statistique).

### 4.5 Une note sur l'effet vs la significativité

> **Point important à toujours mentionner dans un rapport pour client** : une p-value significative ne dit **rien** sur l'**ampleur** de l'effet. Avec n ≈ 1000 patients, un écart d'âge de 2 ans entre sous-types peut être statistiquement très significatif mais cliniquement négligeable.
>
> Toujours compléter une p-value par une **taille d'effet** (différence de médianes, V de Cramér, η²) et un **intervalle de confiance**.


In [ ]:
# Exemple : différence de médianes d'âge entre LumA et Basal, avec IC bootstrap
if 'pam50_subtype' in clinical.columns and {'BRCA_LumA', 'BRCA_Basal'}.issubset(set(clinical['pam50_subtype'].dropna())):
    luma = clinical.loc[clinical['pam50_subtype'] == 'BRCA_LumA', 'age_years'].dropna().values
    basal = clinical.loc[clinical['pam50_subtype'] == 'BRCA_Basal', 'age_years'].dropna().values

    obs_diff = np.median(luma) - np.median(basal)

    # Bootstrap (10 000 réplications)
    n_boot = 10_000
    boot_diffs = np.empty(n_boot)
    rng = np.random.default_rng(RANDOM_STATE)
    for i in range(n_boot):
        b_luma = rng.choice(luma, size=len(luma), replace=True)
        b_basal = rng.choice(basal, size=len(basal), replace=True)
        boot_diffs[i] = np.median(b_luma) - np.median(b_basal)

    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    print(f"Différence de médianes (LumA − Basal) : {obs_diff:+.1f} ans")
    print(f"IC 95% bootstrap : [{ci_low:+.1f}, {ci_high:+.1f}] ans")


<a id="5-viz"></a>
## 5. Visualisations de synthèse

> **Principe directeur** : ces figures sont destinées à un **clinicien** ou un **décideur biotech**, pas à un statisticien. Elles doivent être lisibles en 5 secondes et raconter l'essentiel.

### 5.1 Figure "hero" — résumé des différences inter-sous-types

Une figure unique qui combine âge, stade et mortalité par sous-type. À retravailler pour ton portfolio.


In [ ]:
if 'pam50_subtype' in clinical.columns and clinical['pam50_subtype'].notna().any():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel A : âge
    sns.boxplot(
        data=clinical.dropna(subset=['pam50_subtype']),
        x='pam50_subtype', y='age_years', order=available_subtypes,
        palette='colorblind', ax=axes[0]
    )
    axes[0].set_title("A. Âge au diagnostic")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Âge (années)")
    axes[0].tick_params(axis='x', rotation=30)

    # Panel B : stade (% stade III-IV)
    advanced_stage = (
        clinical.dropna(subset=['pam50_subtype', 'stage_simplified'])
        .assign(advanced=lambda d: d['stage_simplified'].isin(['III', 'IV']))
        .groupby('pam50_subtype')['advanced'].mean() * 100
    ).reindex(available_subtypes)

    sns.barplot(x=advanced_stage.index, y=advanced_stage.values, palette='colorblind', ax=axes[1])
    axes[1].set_title("B. % diagnostics stade III-IV")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("% patients")
    axes[1].tick_params(axis='x', rotation=30)

    # Panel C : mortalité observée
    mortality = (
        clinical.dropna(subset=['pam50_subtype'])
        .assign(dead=lambda d: d['vital_status'] == 'Dead')
        .groupby('pam50_subtype')['dead'].mean() * 100
    ).reindex(available_subtypes)

    sns.barplot(x=mortality.index, y=mortality.values, palette='colorblind', ax=axes[2])
    axes[2].set_title("C. % décès au dernier suivi")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("% patients")
    axes[2].tick_params(axis='x', rotation=30)

    fig.suptitle("Caractéristiques cliniques par sous-type PAM50 (TCGA-BRCA)", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'hero_subtype_summary.png', bbox_inches='tight', dpi=300)
    plt.show()


### 5.2 Heatmap de complétude des données

Une figure souvent oubliée mais très appréciée des clients : où sont les trous dans le jeu de données ? C'est une preuve de rigueur scientifique.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
missing_matrix = clinical[variables_of_interest[1:]].isna().T  # exclure l'ID
sns.heatmap(missing_matrix, cbar=False, cmap='RdYlGn_r', ax=ax)
ax.set_title("Carte des données manquantes (rouge = manquant)")
ax.set_xlabel("Patients (ordonnés)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_missingness_heatmap.png', bbox_inches='tight')
plt.show()


<a id="6-discussion"></a>
## 6. Discussion, limites et perspectives

### 6.1 Principaux résultats

> *À rédiger après exécution complète, en 3-5 puces, en langage clinique.*
>
> Exemple de structure :
> - Le sous-type Basal-like est diagnostiqué à un âge significativement plus jeune (médiane X ans vs Y ans pour Luminal A, p_FDR = ...)
> - Le sous-type Basal-like est sur-représenté dans les stades avancés (Z% vs W% pour Luminal A)
> - ...

### 6.2 Limites méthodologiques

À identifier et **toujours mentionner pour un client** :

1. **Biais de sélection TCGA** — la cohorte TCGA n'est pas représentative de la population générale (centres académiques américains, critères d'inclusion stricts). Les conclusions ne s'extrapolent pas directement à d'autres populations.

2. **Données manquantes non aléatoires** — certains champs (statut HER2, traitements) ont des taux de NaN > 30%. Les patients avec données complètes diffèrent peut-être des autres.

3. **Mortalité ≠ survie** — la comparaison `vital_status` sans tenir compte du temps de suivi est un proxy grossier. Le **Projet 4** apportera une analyse de survie rigoureuse (Kaplan-Meier, Cox).

4. **Pas de validation externe** — toutes les analyses portent sur un seul jeu de données. Idéalement, on confirmerait sur une cohorte indépendante (METABRIC par exemple).

5. **Tests multiples** — seulement 3 tests ici, mais en pratique on en fait souvent des dizaines. La correction FDR est appliquée mais reste une approximation.

### 6.3 Prochaines étapes (Projets 2-4)

- **Projet 2** : analyse différentielle d'expression génique entre sous-types Basal-like et Luminal A → identifier les gènes différentiellement exprimés et leur cohérence avec la littérature.
- **Projet 3** : dashboard interactif Streamlit permettant à un clinicien de filtrer la cohorte et d'explorer ces analyses sans coder.
- **Projet 4** : analyse de survie (Kaplan-Meier par sous-type, modèle de Cox ajusté sur l'âge et le stade) pour quantifier réellement le pronostic différentiel.

### 6.4 Reproductibilité

- **Code** : disponible sur GitHub à l'adresse [URL]
- **Données** : TCGA-BRCA, accessible via portal.gdc.cancer.gov (compte gratuit)
- **Environnement** : `requirements.txt` fourni dans le repo
- **Seed** : `RANDOM_STATE = 42` pour toutes les opérations stochastiques

---

## Références

1. The Cancer Genome Atlas Network. *Comprehensive molecular portraits of human breast tumours.* Nature 490, 61–70 (2012).
2. Parker JS et al. *Supervised risk predictor of breast cancer based on intrinsic subtypes (PAM50).* JCO 27 (2009).
3. Benjamini Y, Hochberg Y. *Controlling the false discovery rate.* J R Stat Soc B 57 (1995).
